# LinguaFranca — Phase 1: Dataset Construction
> **Do Internal Failures Predict External Ones?**

This notebook runs the full Phase 1 pipeline:
1. Install dependencies
2. Clone source code from GitHub
3. Configure for Kaggle (model, paths, scale)
4. Download 2WikiMultihopQA + HotpotQA
5. Generate XML-tagged CoT trajectories
5b. *(Pilot only)* Probe hidden states per layer → pick best layers
6. Label each hop as success/failure
7. Build counterfactual examples
8. Write `train/val/test.jsonl` splits
9. Inspect a sample

**GPU required** — set accelerator to GPU (T4 x2 or P100) in Notebook Settings.

---
### Chat prompt format
`build_prompt()` uses `tokenizer.apply_chat_template()` so the format is
automatically correct for whichever model is loaded (Qwen2.5-Instruct,
Llama-3.2-Instruct, etc.) — no hardcoded special tokens.

### Layer selection workflow
Run **Step 5** with `n_source_examples=200` and `layers="all"` (the pilot).
Then run **Step 5b** to find the best layers. Lock them in, bump
`n_source_examples` to 2000, and re-run Step 5 for the full extraction.

## Step 0 — Check GPU

In [ ]:
import subprocess
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                        capture_output=True, text=True)
print('GPU:', result.stdout.strip() or 'NOT FOUND — enable GPU in Settings!')

## Step 1 — Install Dependencies

In [ ]:
%%capture
!pip install nnsight sentence-transformers SPARQLWrapper datasets accelerate -q

## Step 2 — Get Source Code from GitHub

This directly clones the latest code from the GitHub repository.

In [ ]:
import os, sys
import subprocess

REPO_URL = "https://github.com/ANLPproject/LinguaFranca.git"
WORK_DIR = "/kaggle/working/LinguaFranca"

if not os.path.exists(WORK_DIR):
    print(f"Cloning repository from {REPO_URL}...")
    subprocess.run(["git", "clone", REPO_URL, WORK_DIR], check=True)
else:
    print("Repository already exists. Pulling latest changes...")
    subprocess.run(["git", "-C", WORK_DIR, "pull"], check=True)

os.chdir(WORK_DIR)
sys.path.insert(0, WORK_DIR)
print('Working directory:', os.getcwd())
print('Contents:', os.listdir('.'))

## Step 3 — Configure for Kaggle

**Two modes:**
- `PILOT = True` — 200 examples, all layers. Cheap. Run this first to find best layers.
- `PILOT = False` — 2000 examples, locked layers only. Run after Step 5b.

In [ ]:
import yaml

# ── Set this flag ──────────────────────────────────────────────────────
PILOT = True   # True = 200-example pilot; False = 2000-example full run

# After running Step 5b on the pilot, set these to your best layers:
BEST_LAYERS = "all"   # e.g. [8, 12, 16, 20, 24, 28] after probing

with open('configs/data_config.yaml') as f:
    cfg = yaml.safe_load(f)

# ── Model ──────────────────────────────────────────────────────────────
# Qwen2.5-Instruct works without an HF token on Kaggle.
# tokenizer.apply_chat_template() handles the correct format automatically.
cfg['model']['name']              = 'Qwen/Qwen2.5-3B-Instruct'
cfg['model']['trust_remote_code'] = True
cfg['model']['dtype']             = 'bfloat16'   # better on A100/T4

# ── Paths ──────────────────────────────────────────────────────────────
cfg['data']['raw_dir']            = '/kaggle/working/data/raw'
cfg['data']['processed_dir']      = '/kaggle/working/data/processed'
cfg['data']['hidden_states_dir']  = '/kaggle/working/data/hidden_states'
cfg['matching']['wikidata_aliases_path'] = '/kaggle/working/data/raw/wikidata_aliases.json'

# ── Scale (pilot vs full run) ──────────────────────────────────────────
if PILOT:
    cfg['data']['n_source_examples']  = 200    # cheap pilot: ~15 min on T4
    cfg['hidden_states']['layers']    = 'all'  # probe all layers on small set
    print('[PILOT MODE] 200 examples, all layers')
else:
    cfg['data']['n_source_examples']  = 2000   # full run after layer selection
    cfg['hidden_states']['layers']    = BEST_LAYERS
    print(f'[FULL MODE] 2000 examples, layers={BEST_LAYERS}')

cfg['generation']['batch_size'] = 4

with open('configs/data_config.yaml', 'w') as f:
    yaml.dump(cfg, f)

print('Config updated:')
print(f"  model      : {cfg['model']['name']}")
print(f"  n_examples : {cfg['data']['n_source_examples']}")
print(f"  layers     : {cfg['hidden_states']['layers']}")
print(f"  raw_dir    : {cfg['data']['raw_dir']}")

## Step 4 — Download Datasets

In [ ]:
import logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s  %(levelname)-8s  %(message)s')

from phase1_dataset.download import download_all
paths = download_all(cfg)
for ds, splits in paths.items():
    for split, p in splits.items():
        print(f'  {ds}/{split} → {p}')

## Step 5 — Generate CoT + Extract Hidden States

> ⏱ **Pilot (200 ex)**: ~15 min on T4  
> ⏱ **Full run (2000 ex)**: ~2 hrs on T4

`build_prompt()` uses `tokenizer.apply_chat_template()` — correct format for any model.

In [ ]:
from phase1_dataset.generate_cot import run_generation
cot_path = run_generation(cfg, dry_run=False)
print(f'CoT saved to: {cot_path}')

## Step 5b — Layer Probing (Pilot Only)

**Run this only after the pilot (200 examples, all layers).**

Train a quick linear probe per layer on the pilot hidden states to find which layers
are informative. You will usually see: early layers useless, a rise through the middle,
a plateau or small dip near the end.

Pick the 5–8 peak layers, set `BEST_LAYERS` in Step 3, flip `PILOT = False`, and re-run.

In [ ]:
if not PILOT:
    print('Skip — only run layer probing on the pilot output.')
else:
    import json, torch, numpy as np
    from pathlib import Path
    from sklearn.linear_model import LogisticRegression
    from sklearn.metrics import accuracy_score

    hs_dir = Path(cfg['data']['hidden_states_dir'])
    cot_file = Path(cfg['data']['raw_dir']) / '2wikimultihopqa' / 'generated_cot.jsonl'
    examples = [json.loads(l) for l in open(cot_file)]
    print(f'Loaded {len(examples)} pilot examples')

    sample_pt = next(hs_dir.glob('*.pt'), None)
    if sample_pt is None:
        print('No .pt files found — re-run Step 5 with extract=true')
    else:
        sample = torch.load(sample_pt)
        n_layers = sample['pooled'].shape[0]
        layer_indices = sample.get('layer_indices', list(range(n_layers)))

        X_by_layer = {li: [] for li in layer_indices}
        y = []
        for ex in examples:
            pt_path = hs_dir / f"{ex['id']}.pt"
            if not pt_path.exists(): continue
            data = torch.load(pt_path)
            pooled = data['pooled']  # [n_layers, n_hops, hidden_dim]
            li_list = data.get('layer_indices', list(range(pooled.shape[0])))
            for li_idx, li in enumerate(li_list):
                X_by_layer[li].append(pooled[li_idx].mean(0).detach().numpy())
            y.append(1 if ex.get('first_fail_hop') is not None else 0)

        y = np.array(y)
        pct = 100 * y.mean()
        print(f'Features: {len(y)} examples  |  failure ratio: {pct:.1f}%')

        layer_accs = {}
        for li in layer_indices:
            X = np.array(X_by_layer[li])
            if len(X) == 0: continue
            X = np.nan_to_num(X)
            if len(np.unique(y)) < 2: continue
            clf = LogisticRegression(max_iter=300)
            clf.fit(X, y)
            layer_accs[li] = accuracy_score(y, clf.predict(X))

        print('\nLayer probe accuracy:')
        for li, acc in sorted(layer_accs.items()):
            bar = chr(0x2588) * int(acc * 40)
            print(f'  Layer {li:2d}: {acc:.3f}  {bar}')

        best = sorted(layer_accs, key=layer_accs.get, reverse=True)[:8]
        print(f'\n-> Suggested BEST_LAYERS = {sorted(best)}')
        print('Set this in Step 3, flip PILOT=False, and re-run Steps 3-5.')

## Step 6 — Label Hops

In [ ]:
from phase1_dataset.label_hops import run_labeling
labeled_path = run_labeling(cfg)
print(f'Labeled data: {labeled_path}')

# Quick stats
import json
with open(labeled_path) as f:
    examples = [json.loads(l) for l in f]

total_hops = sum(len(e['hops']) for e in examples)
fail_hops  = sum(1 for e in examples for h in e['hops'] if h['label'] == 1)
print(f'Total hops: {total_hops}  |  Failures: {fail_hops} ({100*fail_hops/total_hops:.1f}%)')

## Step 7 — Build Counterfactuals (class balancing)

In [ ]:
from phase1_dataset.counterfactuals import run_counterfactuals
augmented_path = run_counterfactuals(cfg)
print(f'Augmented data: {augmented_path}')

## Step 8 — Split and Write Final JSONL Files

In [ ]:
import random
from phase1_dataset.build_dataset import split_by_id, check_class_balance, check_no_leakage, save_jsonl, sample_hotpotqa
from pathlib import Path

with open(augmented_path) as f:
    all_examples = [json.loads(l) for l in f]

rng = random.Random(cfg.get('seed', 42))
splits = cfg['data']['splits']
train, val, test = split_by_id(all_examples, splits['train'], splits['val'], rng)

# Checks
print('Class balance:')
check_class_balance('train', train)
check_class_balance('val',   val)
check_class_balance('test',  test)
check_no_leakage(train, val, test)

# Write
processed = Path(cfg['data']['processed_dir'])
save_jsonl(train, processed / 'train.jsonl')
save_jsonl(val,   processed / 'val.jsonl')
save_jsonl(test,  processed / 'test.jsonl')

# HotpotQA OOD
hotpot = sample_hotpotqa(cfg, rng)
if hotpot:
    save_jsonl(hotpot, processed / 'hotpotqa_test.jsonl')

print(f'\nDone!  train={len(train)} | val={len(val)} | test={len(test)}')

## Step 9 — Inspect a Sample

In [ ]:
import json, random

with open(processed / 'train.jsonl') as f:
    train_data = [json.loads(l) for l in f]

# Show 3 random examples
for ex in random.sample(train_data, 3):
    print('─' * 60)
    print(f"Q: {ex['question']}")
    print(f"Gold answer: {ex['gold_answer']}")
    print(f"Counterfactual: {ex.get('is_counterfactual', False)}")
    for h in ex['hops']:
        status = '✓' if h['label'] == 0 else '✗'
        print(f"  hop{h['hop_idx']} [{status}] ({h['match_method']}) gold={h['bridging_entity_gold']!r}")
        print(f"         text: {h['text'][:80]}...")
    print(f"  first_fail_hop: {ex['first_fail_hop']}")

## ✅ Phase 1 Complete!

Output files are in `/kaggle/working/data/processed/`.
Download them via **Data → Output** in the Kaggle sidebar.

### Next steps
- **If this was a pilot run:** Check Step 5b for best layers, then re-run with `PILOT=False`.
- **If this was the full run:** Proceed to **Phase 2 — Train the hop-failure probe** on `train.jsonl`
  using the hidden states in `/kaggle/working/data/hidden_states/`.